In [1]:
import numpy as np
import pandas as pd
import os
import glob
import itertools
import scanpy as sc
import natsort
import json

import matplotlib.pyplot as plt
import seaborn as sns

from scroutines import basicu
from scroutines import powerplots

import scanpy.external as sce


In [2]:
ddir = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome'
outdir = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/astro_john/scenicplus_inputdata'
!ls $ddir/*.h5ad
# !ls $outdir

/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/L23_allmultiome_proc_P6toP21.h5ad
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/L23_allmultiome_proc_P6toP21_NRDR.h5ad
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/L23_allmultiome_raw.h5ad
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/L23_allmultiome_raw_pca.h5ad
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/L23_Allraw.h5ad
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/multiome_dpt_P10.h5ad
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/multiome_dpt_P12DR.h5ad
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/multiome_dpt_P12.h5ad
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/multiome_dpt_P14DR.h5ad
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/multiome_dpt_P14.h5ad
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/multiome_dpt_P17DR.h5ad
/u/home/f/f7xiesnm/project-zipursky/v

In [3]:
f = "/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/superdupermegaRNA_hasraw_multiome_P21NRDR.h5ad"
print(f)
adata = sc.read(f)
print(adata)

/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/superdupermegaRNA_hasraw_multiome_P21NRDR.h5ad
AnnData object with n_obs × n_vars = 31039 × 16572
    obs: 'Age', 'Doublet', 'Doublet Score', 'n_counts', 'n_genes', 'percent_mito', 'sample', 'Type', 'Subclass', 'Class', 'Sample', 'total_counts', 'pct_counts_mt', 'n_genes_by_counts', 'total_counts_mt', 'Doublet?', 'Study', 'Type_leiden'
    var: 'feature_types'


In [4]:
adata.raw.X.data

array([35.,  2.,  1., ...,  3.,  8.,  1.], dtype=float32)

In [5]:
adata.X.data

array([0.38399825, 0.38399825, 1.0552076 , ..., 1.4792906 , 1.4792906 ,
       2.887543  ], dtype=float32)

In [6]:
adata.X = adata.raw.X

In [7]:
adata.obs.Age.unique()

['P21', 'P21DR']
Categories (2, object): ['P21', 'P21DR']

In [8]:
counts = adata.obs.Subclass.value_counts()
counts

L2/3      6469
L4        5931
L6CT      3421
Astro     2142
OD        1802
Micro     1786
L6IT      1718
OPC       1269
L5IT      1211
L5PT      1154
Pvalb     1063
Sst        789
Vip        500
L5NP       495
Lamp5      389
L6b        268
L2/3/4     214
Endo       170
VLMC       105
Frem1       84
Stac        59
Name: Subclass, dtype: int64

In [9]:
num_cells_cutoff=2000
sel_types = counts.index.values[counts > num_cells_cutoff]
adata = adata[adata.obs['Subclass'].isin(sel_types)]
adata.obs['Subclass'].unique()

['L2/3', 'L4', 'L6CT', 'Astro']
Categories (4, object): ['Astro', 'L2/3', 'L4', 'L6CT']

In [10]:
def streamline_barcode(x):
    return x.split(' ')[0][:-len("-2023")]


cell_barcodes = adata.obs_names.values
vfunc  = np.vectorize(streamline_barcode)
cell_barcodes_simp = vfunc(cell_barcodes)

assert len(np.unique(cell_barcodes_simp)) == len(cell_barcodes)
cell_barcodes_simp

array(['AAACCGAAGTTCCTGC-1-P21a', 'AAACGCGCAATCATGT-1-P21a',
       'AAACGCGCAGTTGCGT-1-P21a', ..., 'CGGACCTAGGCTTAAC-1-P21DRa',
       'GAGGGAGCATAATGAG-1-P21DRb', 'GAGAAACGTTAGCATG-1-P21DRa'],
      dtype='<U25')

In [11]:
adata.obs_names = cell_barcodes_simp

In [12]:
adata.obs

,Age,Doublet,Doublet Score,n_counts,n_genes,percent_mito,sample,Type,Subclass,Class,Sample,total_counts,pct_counts_mt,n_genes_by_counts,total_counts_mt,Doublet?,Study,Type_leiden
AAACCGAAGTTCCTGC-1-P21a,P21,False,0.011412,NaN,NaN,NaN,NaN,L2/3_B,L2/3,Excitatory,P21a,21809.0,0.389747,4625.0,85.0,0.0,2023 Multiome,L2/3_B
AAACGCGCAATCATGT-1-P21a,P21,False,0.034908,NaN,NaN,NaN,NaN,L2/3_B,L2/3,Excitatory,P21a,26366.0,0.056891,5289.0,15.0,0.0,2023 Multiome,L2/3_B
AAACGCGCAGTTGCGT-1-P21a,P21,False,0.045970,NaN,NaN,NaN,NaN,L2/3_B,L2/3,Excitatory,P21a,14682.0,0.224765,4483.0,33.0,0.0,2023 Multiome,L2/3_B
AAACGCGCATTGTGTG-1-P21a,P21,False,0.043211,NaN,NaN,NaN,NaN,L2/3_B,L2/3,Excitatory,P21a,19590.0,0.653395,5066.0,128.0,0.0,2023 Multiome,L2/3_B
AAAGCAAGTTGACTTC-1-P21a,P21,False,0.022759,NaN,NaN,NaN,NaN,L2/3_B,L2/3,Excitatory,P21a,10958.0,0.447162,3764.0,49.0,0.0,2023 Multiome,L2/3_B
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GGTGAGCCACCAACCG-1-P21DRa,P21DR,False,0.011288,NaN,NaN,NaN,NaN,Astro_Fem2,Astro,Non-neurons,P21DRa,2554.0,4.581049,1518.0,117.0,0.0,2023 Multiome,Astro_Fem2
GGTTGAGCACTAAGCC-1-P21DRb,P21DR,False,0.010065,NaN,NaN,NaN,NaN,Astro_Fem2,Astro,Non-neurons,P21DRb,3263.0,3.708244,1780.0,121.0,0.0,2023 Multiome,Astro_Fem2
CGGACCTAGGCTTAAC-1-P21DRa,P21DR,False,0.004323,NaN,NaN,NaN,NaN,Astro_Fem1,Astro,Non-neurons,P21DRa,1647.0,1.578628,1009.0,26.0,0.0,2023 Multiome,Astro_Fem1
GAGGGAGCATAATGAG-1-P21DRb,P21DR,False,0.007982,NaN,NaN,NaN,NaN,Astro_Fem2,Astro,Non-neurons,P21DRb,3465.0,8.542568,1899.0,296.0,0.0,2023 Multiome,Astro_Fem2


In [13]:
adata.raw.X.shape #data

(17963, 16572)

In [14]:
rna = adata

In [15]:
np.random.seed(0)

# Basic QC
sc.pp.filter_cells(rna, min_genes=200)
sc.pp.filter_genes(rna, min_cells=3)
rna.var["mt"] = rna.var_names.str.startswith("mt-")  # mouse uses lowercase "mt-"
sc.pp.calculate_qc_metrics(rna, qc_vars=["mt"], inplace=True)
rna = rna[rna.obs.pct_counts_mt < 20].copy()

# Normalise & cluster
sc.pp.normalize_total(rna, target_sum=1e4)
sc.pp.log1p(rna)
sc.pp.highly_variable_genes(rna, n_top_genes=3000)
rna.raw = rna
rna = rna[:, rna.var.highly_variable].copy()
sc.pp.scale(rna, max_value=10)
sc.tl.pca(rna, n_comps=50)
sc.pp.neighbors(rna)
sc.tl.umap(rna)
sc.tl.leiden(rna, resolution=0.5)

rna.write_h5ad(os.path.join(outdir, "rna_preprocessed_astro_yoo25_v2.h5ad"))
print(f"  RNA: {rna.shape[0]} cells × {rna.shape[1]} genes")

  RNA: 17963 cells × 3000 genes


# write barcodes

In [16]:
cell_barcodes = rna.obs_names.values
cell_barcodes

array(['AAACCGAAGTTCCTGC-1-P21a', 'AAACGCGCAATCATGT-1-P21a',
       'AAACGCGCAGTTGCGT-1-P21a', ..., 'CGGACCTAGGCTTAAC-1-P21DRa',
       'GAGGGAGCATAATGAG-1-P21DRb', 'GAGAAACGTTAGCATG-1-P21DRa'],
      dtype=object)

In [17]:
def streamline_barcode2(x):
    return '-'.join(x.split('-')[:2])

def streamline_barcode3(x):
    return x.split('-')[2]

vfunc2 = np.vectorize(streamline_barcode2)
vfunc3 = np.vectorize(streamline_barcode3)

cell_barcodes_nosamp = vfunc2(cell_barcodes)
cell_barcodes_samp   = vfunc3(cell_barcodes)

In [18]:
uniqs, counts = np.unique(cell_barcodes_simp, return_counts=True)
np.any(counts > 1)

False

In [19]:
cell_barcodes_nosamp

array(['AAACCGAAGTTCCTGC-1', 'AAACGCGCAATCATGT-1', 'AAACGCGCAGTTGCGT-1',
       ..., 'CGGACCTAGGCTTAAC-1', 'GAGGGAGCATAATGAG-1',
       'GAGAAACGTTAGCATG-1'], dtype='<U18')

In [20]:
uniq_samps = np.unique(cell_barcodes_samp)
uniq_samps

array(['P21DRa', 'P21DRb', 'P21a', 'P21b'], dtype='<U6')

In [21]:
# Write barcodes to file first
for samp in uniq_samps:
    sel_cond = cell_barcodes_samp == samp
    sel_barcodes = cell_barcodes_nosamp[sel_cond]
    
    fout = f'/u/home/f/f7xiesnm/astro_john/scenicplus_inputdata/barcodes/yoo25_astro_cell_barcodes_v2_{samp}.txt'
    print(fout)
    with open(fout, "w") as f:
        f.write("\n".join(sel_barcodes))

/u/home/f/f7xiesnm/astro_john/scenicplus_inputdata/barcodes/yoo25_astro_cell_barcodes_v2_P21DRa.txt
/u/home/f/f7xiesnm/astro_john/scenicplus_inputdata/barcodes/yoo25_astro_cell_barcodes_v2_P21DRb.txt
/u/home/f/f7xiesnm/astro_john/scenicplus_inputdata/barcodes/yoo25_astro_cell_barcodes_v2_P21a.txt
/u/home/f/f7xiesnm/astro_john/scenicplus_inputdata/barcodes/yoo25_astro_cell_barcodes_v2_P21b.txt
